In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
try:
    import folium
except ModuleNotFoundError:
    import subprocess
    import sys
    import importlib

    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])
    importlib.invalidate_caches()
    import folium
from folium.plugins import MarkerCluster

In [5]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

In [6]:
print("=" * 75)
print("GEOSPATIAL DATA ANALYSIS - SUPERSTORE")
print("=" * 75)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

GEOSPATIAL DATA ANALYSIS - SUPERSTORE

First 5 rows:
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID 

In [7]:
df.describe(include="all")

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
count,9994.000000,9994,9994,9994,9994,9994,9994,9994,9994,9994,...,9994.000000,9994,9994,9994,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000
unique,NaN,5009,1237,1334,4,793,793,3,1,531,...,NaN,4,1862,3,17,1850,NaN,NaN,NaN,NaN
top,NaN,CA-2017-100111,9/5/2016,12/16/2015,Standard Class,WB-21850,William Brown,Consumer,United States,New York City,...,NaN,West,OFF-PA-10001970,Office Supplies,Binders,Staple envelope,NaN,NaN,NaN,NaN
freq,NaN,14,38,35,5968,37,37,5191,9994,915,...,NaN,3203,19,6026,1523,48,NaN,NaN,NaN,NaN
mean,4997.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,55190.379428,NaN,NaN,NaN,NaN,NaN,229.858001,3.789574,0.156203,28.656896
std,2885.163629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,32063.693350,NaN,NaN,NaN,NaN,NaN,623.245101,2.225110,0.206452,234.260108
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1040.000000,NaN,NaN,NaN,NaN,NaN,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23223.000000,NaN,NaN,NaN,NaN,NaN,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,56430.500000,NaN,NaN,NaN,NaN,NaN,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,90008.000000,NaN,NaN,NaN,NaN,NaN,209.940000,5.000000,0.200000,29.364000


In [8]:
# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows with missing location information
df = df.dropna(
    subset=["State", "City", "Postal Code", "Sales"]
)

In [9]:
df["Postal Code"] = (
    df["Postal Code"]
    .astype(int)
    .astype(str)
)

# Convert Sales to numeric
df["Sales"] = pd.to_numeric(
    df["Sales"],
    errors="coerce"
)

In [10]:
# Remove invalid sales
df = df[df["Sales"] >= 0]

print("\nCleaned Dataset Shape:")
print(df.shape)



Cleaned Dataset Shape:
(9994, 21)


In [11]:
state_summary = (
    df.groupby("State")
    .agg(
        Revenue=("Sales", "sum"),
        Orders=("Order ID", "nunique"),
        Customers=("Customer ID", "nunique"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

# Average revenue per customer
state_summary["Revenue_per_Customer"] = (
    state_summary["Revenue"] /
    state_summary["Customers"]
)

In [12]:
city_summary = (
    df.groupby(
        ["State", "City", "Postal Code"]
    )
    .agg(
        Revenue=("Sales", "sum"),
        Orders=("Order ID", "nunique"),
        Customers=("Customer ID", "nunique"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

city_summary["Revenue_per_Customer"] = (
    city_summary["Revenue"] /
    city_summary["Customers"]
)

In [13]:
print("\n" + "=" * 75)
print("TOP 10 STATES BY REVENUE")
print("=" * 75)

top_states = (
    state_summary
    .sort_values("Revenue", ascending=False)
    .head(10)
)

print(top_states.to_string(index=False))


TOP 10 STATES BY REVENUE
       State     Revenue  Orders  Customers  Quantity  Revenue_per_Customer
  California 457687.6315    1021        577      7667            793.219465
    New York 310876.2710     562        415      4224            749.099448
       Texas 170188.0458     487        370      3724            459.967691
  Washington 138641.2700     256        224      1883            618.934241
Pennsylvania 116511.9140     288        257      2153            453.353751
     Florida  89473.7080     200        181      1379            494.329878
    Illinois  80166.1010     276        237      1845            338.253591
        Ohio  78258.1360     236        202      1759            387.416515
    Michigan  76269.6140     117        106       946            719.524660
    Virginia  70636.7200     115        107       893            660.156262


In [14]:
print("\nCreating state revenue map...")

fig = px.choropleth(
    state_summary,
    locations="State",
    locationmode="USA-states",
    color="Revenue",
    scope="usa",
    hover_name="State",
    hover_data=[
        "Revenue",
        "Orders",
        "Customers",
        "Revenue_per_Customer"
    ],
    color_continuous_scale="Viridis",
    title="US Revenue by State"
)

fig.update_layout(
    geo=dict(
        showlakes=True,
        lakecolor="white"
    )
)

fig.write_html(
    "state_revenue_choropleth.html"
)

fig.show()


Creating state revenue map...


In [15]:
fig3 = px.scatter(
    state_summary,
    x="Customers",
    y="Revenue",
    size="Orders",
    hover_name="State",
    title="Revenue vs Customer Density",
    labels={
        "Customers": "Number of Customers",
        "Revenue": "Total Revenue"
    }
)

fig3.write_html(
    "revenue_vs_customer_density.html"
)

fig3.show()

In [16]:
# Calculate median values
median_revenue = state_summary["Revenue"].median()
median_customers = state_summary["Customers"].median()

# Define underserved:
# High customer demand but comparatively low revenue

state_summary["Underserved"] = np.where(
    (state_summary["Customers"] > median_customers) &
    (state_summary["Revenue"] < median_revenue),
    "Yes",
    "No"
)

underserved = state_summary[
    state_summary["Underserved"] == "Yes"
].copy()

In [17]:
# Calculate opportunity score
underserved["Opportunity_Score"] = (
    underserved["Customers"] /
    underserved["Customers"].max()
    +
    (1 -
     underserved["Revenue"] /
     underserved["Revenue"].max())
)

top_3_underserved = (
    underserved
    .sort_values(
        "Opportunity_Score",
        ascending=False
    )
    .head(3)
)

print("\n" + "=" * 75)
print("TOP 3 HIGH-POTENTIAL UNDERSERVED REGIONS")
print("=" * 75)

print(
    top_3_underserved[
        [
            "State",
            "Revenue",
            "Customers",
            "Orders",
            "Revenue_per_Customer",
            "Opportunity_Score"
        ]
    ].to_string(index=False)
)



TOP 3 HIGH-POTENTIAL UNDERSERVED REGIONS
      State   Revenue  Customers  Orders  Revenue_per_Customer  Opportunity_Score
Connecticut 13384.357         43      45            311.264116           1.075296
     Oregon 17431.150         51      56            341.787255           1.000000


In [18]:
state_summary.to_csv(
    "state_analysis.csv",
    index=False
)

city_summary.to_csv(
    "city_analysis.csv",
    index=False
)

top_3_underserved.to_csv(
    "top_3_underserved_regions.csv",
    index=False
)

In [19]:
locations = {
    "New York": [40.7128, -74.0060],
    "California": [36.7783, -119.4179],
    "Texas": [31.9686, -99.9018],
    "Florida": [27.6648, -81.5158],
    "Illinois": [40.6331, -89.3985],
    "Pennsylvania": [41.2033, -77.1945],
    "Ohio": [40.4173, -82.9071],
    "Georgia": [32.1656, -82.9001],
    "North Carolina": [35.7596, -79.0193],
    "Michigan": [44.3148, -85.6024],
    "Virginia": [37.4316, -78.6569],
    "Washington": [47.4009, -121.4905],
    "Arizona": [34.0489, -111.0937],
    "Colorado": [39.5501, -105.7821],
    "Tennessee": [35.5175, -86.5804],
    "Indiana": [40.2672, -86.1349],
    "Massachusetts": [42.4072, -71.3824],
    "Missouri": [37.9643, -91.8318],
    "Maryland": [39.0458, -76.6413],
    "Wisconsin": [43.7844, -88.7879]
}

# Create base map
m = folium.Map(
    location=[39.8283, -98.5795],
    zoom_start=4
)

marker_cluster = MarkerCluster().add_to(m)

# Add state markers
for _, row in state_summary.iterrows():

    state = row["State"]

    if state in locations:

        lat, lon = locations[state]

        popup_text = f"""
        <b>State:</b> {state}<br>
        <b>Revenue:</b> ${row['Revenue']:,.2f}<br>
        <b>Customers:</b> {row['Customers']:,}<br>
        <b>Orders:</b> {row['Orders']:,}
        """

        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(
                popup_text,
                max_width=300
            ),
            tooltip=state
        ).add_to(marker_cluster)


In [20]:
for _, row in top_3_underserved.iterrows():

    state = row["State"]

    if state in locations:

        lat, lon = locations[state]

        popup_text = f"""
        <b>HIGH-POTENTIAL UNDERSERVED REGION</b><br><br>
        <b>State:</b> {state}<br>
        <b>Revenue:</b> ${row['Revenue']:,.2f}<br>
        <b>Customers:</b> {row['Customers']:,}<br>
        <b>Orders:</b> {row['Orders']:,}<br>
        <b>Opportunity Score:</b>
        {row['Opportunity_Score']:.3f}
        """

        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(
                popup_text,
                max_width=350
            ),
            tooltip=f"⭐ High Potential: {state}",
            icon=folium.Icon(
                icon="star",
                prefix="fa"
            )
        ).add_to(m)

# Save map
m.save(
    "interactive_geospatial_map.html"
)

print("\nInteractive map saved as:")
print("interactive_geospatial_map.html")


Interactive map saved as:
interactive_geospatial_map.html


In [21]:
print("\n" + "=" * 75)
print("BUSINESS INSIGHTS")
print("=" * 75)

for index, row in top_3_underserved.iterrows():

    print(
        f"""
{index + 1}. {row['State']}
   Customers       : {row['Customers']:,}
   Revenue         : ${row['Revenue']:,.2f}
   Orders          : {row['Orders']:,}
   Revenue/Customer: ${row['Revenue_per_Customer']:,.2f}
   Opportunity Score: {row['Opportunity_Score']:.3f}
"""
    )

print("=" * 75)
print("GEOSPATIAL ANALYSIS COMPLETED")
print("=" * 75)


BUSINESS INSIGHTS

6. Connecticut
   Customers       : 43
   Revenue         : $13,384.36
   Orders          : 45
   Revenue/Customer: $311.26
   Opportunity Score: 1.075


36. Oregon
   Customers       : 51
   Revenue         : $17,431.15
   Orders          : 56
   Revenue/Customer: $341.79
   Opportunity Score: 1.000

GEOSPATIAL ANALYSIS COMPLETED
